# RAG Document Q&A (LangChain + ChromaDB)

A **retrieval-augmented generation** pipeline that answers questions grounded in a specific document set, so responses are based on source material rather than the model's memory.

**What it demonstrates**
- Chunking and embedding documents into a Chroma vector store
- Retrieving relevant context and composing it into grounded answers
- Using LangChain's current `create_retrieval_chain` pattern (updated from deprecated APIs)

**Stack:** Python · LangChain · ChromaDB · OpenAI embeddings


In [2]:
print("Installing necessary libraries...")
!pip install -q langchain langchain-openai chromadb python-dotenv tiktoken langchain-community
print("Libraries installed successfully!")

Installing necessary libraries...
Libraries installed successfully!


In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key = openai_api_key)

print("OpenAI client initialized successfully!")


OpenAI client initialized successfully!


In [7]:
!pip install langchain-chroma langchain-text-splitters langchain-community

In [2]:
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_chroma import Chroma                                    # was: langchain.vectorstores
from langchain_text_splitters import RecursiveCharacterTextSplitter   # was: langchain.text_splitter
from langchain_community.document_loaders import TextLoader           # was: langchain.document_loaders
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [3]:
DATA_FILE_PATH = "eleven_madison_park_data.txt"
print(f"Data file path set to: {DATA_FILE_PATH}")

Data file path set to: eleven_madison_park_data.txt


In [4]:
loader = TextLoader(DATA_FILE_PATH)

raw_documents = loader.load()
print(f"Successfully loaded {len(raw_documents)} document(s).")

Successfully loaded 1 document(s).


In [5]:
print(raw_documents[0].page_content[:500] + "...")

Source: https://www.elevenmadisonpark.com/
Title: Eleven Madison Park
Content:
Book on Resy
---END OF SOURCE---

Source: https://www.elevenmadisonpark.com/careers
Title: Careers — Eleven Madison Park
Content:
Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Acco...


In [6]:
print("\nSplitting documents into chunks...")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap=150)

documents = text_splitter.split_documents(raw_documents)

if not documents:
    raise ValueError("No documents were created after splitting. Please check the input data and splitting parameters.")
print(f"Successfully split into {len(documents)} chunks.")



Splitting documents into chunks...
Successfully split into 38 chunks.


In [7]:
documents


[Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/\nTitle: Eleven Madison Park\nContent:\nBook on Resy\n---END OF SOURCE---'),
 Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Source: https://www.elevenmadisonpark.com/careers\nTitle: Careers — Eleven Madison Park\nContent:'),
 Document(metadata={'source': 'eleven_madison_park_data.txt'}, page_content="Join Our Team Eleven Madison Park ▾ All Businesses Eleven Madison Park Clemente Bar Daniel Humm Hospitality Filter Categories Culinary Pastry Wine & Beverage Dining Room Office & Admin Other Job Types Full Time Part Time Compensation Salary Hourly Apply filters OPEN OPPORTUNITIES Staff Accountant - Part Time Eleven Madison Park Part Time • Hourly ($20 - $25) Host/Reservationist Eleven Madison Park Full Time • Hourly ($24) Sous Chef Eleven Madison Park Full Time • Salary ($72K - $75K) Pastry Cook Eleven Madison Park Full Time • Hourly ($18 - $2

In [8]:
print("\n--- Example Chunk (Chunk 2) ---")
print(documents[-1].page_content)
print("\n--- Metadata for Chunk 2 ---")
print(documents[-1].metadata) 


--- Example Chunk (Chunk 2) ---
Source: https://www.elevenmadisonpark.com/contact
Title: Contact — Eleven Madison Park
Content:
Contact Us Visit Resy Note: We open up reservations on the first of every month for the following month - for example, on October 1st, all of November will be made available. If you find that the reservation you were hoping for is booked, we do hold a waitlist and encourage you to add your name to it, and if something does become available, we will be in touch. Please contact careers@elevenmadisonpark.com or visit Culinary Agents . To stay up to date about future Eleven Madison Park news and events, please sign up for our Newsletter . Please contact events@elevenmadisonpark.com Please contact press@elevenmadisonpark.com Thank you for thinking of Eleven Madison Park for your inquiry. At this time, all of our non-profit efforts are focused on our partnership with Rethink Food . Email: info@elevenmadisonpark.com Phone: 212.889.0905 Ext. 3
---END OF SOURCE---

--

In [9]:
print("Initialising OpenAI Embeddings...")

embeddings = OpenAIEmbeddings(openai_api_key = openai_api_key)

print("OpenAI Embeddings model initialized!")

print("\nCreating Chroma vector store and embedding documents...")

vector_store = Chroma.from_documents(documents = documents, embedding = embeddings)

vector_count = vector_store._collection.count()
print(f"ChromaDB Vector store created with {vector_count} items.")

if vector_count == 0:
    raise ValueError("No vectors were created. Please check the document splitting and embedding process.")

Initialising OpenAI Embeddings...
OpenAI Embeddings model initialized!

Creating Chroma vector store and embedding documents...
ChromaDB Vector store created with 38 items.


In [10]:
stored_data = vector_store._collection.get(include = ["embeddings", "documents"], limit=1)

print("First chunk text:\n", stored_data["documents"][0])
print("Embedding vector:\n", stored_data["embeddings"][0])
print(f"\nFull embedding has {len(stored_data['embeddings'][0])} dimensions.")
#print(f"\nFull embedding has {len(stored_data['embeddings'][0])} dimensions.")

First chunk text:
 Source: https://www.elevenmadisonpark.com/
Title: Eleven Madison Park
Content:
Book on Resy
---END OF SOURCE---
Embedding vector:
 [ 0.02320839 -0.01570945 -0.00702683 ... -0.02469996 -0.0102563
 -0.06152412]

Full embedding has 1536 dimensions.


In [11]:
print("\n...Testing similarity search in Vector store...\n")
test_query = "What different menus are offered?"
print(f"Searching for documents similar to query: '{test_query}'\n")

try:
    similar_docs = vector_store.similarity_search(test_query, k = 2)
    print(f"Found {len(similar_docs)}  similar documents.")

    for i, doc in enumerate(similar_docs):
        print(f"\n--- Similar Document {i+1} ---")

        content_snippet = doc.page_content[:700].strip() + "..."
        source = doc.metadata.get("source", "Unknown Source")
        print(f"Content Snippet:\n{content_snippet}\n")
        print(f"Source: {source}\n")

except Exception as e:
    print(f"An error occurred during similarity search: {e}")


...Testing similarity search in Vector store...

Searching for documents similar to query: 'What different menus are offered?'

Found 2  similar documents.

--- Similar Document 1 ---
Content Snippet:
FAQs We are located at 11 Madison Avenue, on the northeast corner of East 24th and Madison Avenue, directly across the street from Madison Square Park. We offer three menus, all 100% plant-based: Full Tasting Menu : An eight- to nine-course experience priced at $365 per guest. This menu typically lasts about two to three hours and features a mix of plated and communal dishes. 5-Course Menu : Priced at $285 per guest, this menu highlights selections from the Full Tasting Menu and lasts approximately two hours. Bar Tasting Menu : Available in our lounge for $225 per guest, this menu includes four to five courses and is designed to last around two hours. Note : These durations are estimates bas...

Source: eleven_madison_park_data.txt


--- Similar Document 2 ---
Content Snippet:
Reservatio

In [18]:
# Install first if needed:
# python -m pip install -U langchain langchain-classic langchain-openai

# --- 1. Imports ---
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


# --- 2. Define the Retriever ---
# The retriever uses the vector store to fetch the top k documents
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("Retriever configured successfully from vector store.")


# --- 3. Define the Language Model ---
# Use ChatOpenAI for modern OpenAI chat models
llm = OpenAI(
    temperature=0,
    openai_api_key=openai_api_key
)

print("OpenAI Chat LLM successfully initialized.")


# --- 4. Define the Prompt ---
# The prompt must contain {context}, because the retrieved documents
# will be inserted there by create_stuff_documents_chain.
system_prompt = """
You are a helpful question-answering assistant.

Use ONLY the information in the provided context to answer the question.
If the answer is not in the context, say you do not know.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


# --- 5. Create the Document Chain ---
# This is the replacement for chain_type="stuff"
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

print("Document chain created successfully.")


# --- 6. Create the Retrieval Chain ---
# This replaces RetrievalQAWithSourcesChain
qa_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("Retrieval chain created successfully.")

Retriever configured successfully from vector store.
OpenAI Chat LLM successfully initialized.
Document chain created successfully.
Retrieval chain created successfully.


In [ ]:
print("\n--- Testing the Full RAG Chain ---")
chain_test_query = "What kind of food does Eleven Madison Park serve?"
print(f"Query: {chain_test_query}")



try:
    result = qa_chain.invoke({"input": chain_test_query})  # ✅ "input" not "question"

    # Print the answer
    print("\n--- Answer ---")
    print(result.get("answer", "No answer generated."))

    # Print source document snippets
    # ✅ "context" not "source_documents"
    if "context" in result:
        print("\n--- Source Document Snippets ---")
        for i, doc in enumerate(result["context"]):
            source = doc.metadata.get("source", "Unknown Source")
            content_snippet = doc.page_content[:250].strip()
            print(f"\nDoc {i+1} | Source: {source}")
            print(content_snippet)

except Exception as e:
    print(f"\nAn error occurred while running the chain: {e}")


--- Testing the Full RAG Chain ---
Query: What kind of food does Eleven Madison Park serve?

--- Answer ---


System: Eleven Madison Park serves a fully plant-based menu, using no animal products. They also offer a full tasting menu, a five-course menu, and a bar menu with à la carte snacks.

--- Source Document Snippets ---

Doc 1 | Source: eleven_madison_park_data.txt
Welcome to Eleven Madison Park Eleven Madison Park is a fine dining restaurant in the heart of New York City. Overlooking Madison Square Park–one of Manhattan’s most beautiful green spaces–we sit at the base of a historic Art Deco building on the cor

Doc 2 | Source: eleven_madison_park_data.txt
Source: https://www.elevenmadisonpark.com/ourrestaurant
Title: About — Eleven Madison Park
Content:

Doc 3 | Source: eleven_madison_park_data.txt
Source: https://www.elevenmadisonpark.com/faq
Title: FAQs — Eleven Madison Park
Content:


In [25]:
# --- Test the Full Chain ---
print("\n--- Testing the Full RAG Chain ---")
chain_test_query = "What kind of food does Eleven Madison Park serve?"
print(f"Query: {chain_test_query}")

# ↓ Set this here — True to see source docs, False to hide them
show_source_documents = False

try:
    result = qa_chain.invoke({"input": chain_test_query})

    print("\n--- Answer ---")
    print(result.get("answer", "No answer generated."))

    if show_source_documents:
        print("\n--- Source Document Snippets ---")
        for i, doc in enumerate(result["context"]):
            source = doc.metadata.get("source", "Unknown Source")
            content_snippet = doc.page_content[:250].strip()
            print(f"\nDoc {i+1} | Source: {source}")
            print(content_snippet)

except Exception as e:
    print(f"\nAn error occurred while running the chain: {e}")


--- Testing the Full RAG Chain ---
Query: What kind of food does Eleven Madison Park serve?

--- Answer ---


System: Eleven Madison Park serves a fully plant-based menu, using no animal products. They also offer a full tasting menu, a five-course menu, and a bar menu with à la carte snacks.


In [21]:
result

{'input': 'What kind of food does Eleven Madison Park serve?',
 'context': [Document(id='7fc01e04-7963-4af7-87cb-48739758149f', metadata={'source': 'eleven_madison_park_data.txt'}, page_content='Welcome to Eleven Madison Park Eleven Madison Park is a fine dining restaurant in the heart of New York City. Overlooking Madison Square Park–one of Manhattan’s most beautiful green spaces–we sit at the base of a historic Art Deco building on the corner of East 24th Street and Madison Avenue. Since opening in 1998, we underwent a full-scale renovation and redesign in the summer of 2017. Chef Daniel Humm has owned the restaurant since 2011, during which time we have evolved considerably in both cuisine and experience. In 2021, we transitioned to a fully plant-based menu, using no animal products. That same year, we partnered with Magic Farms , which grows produce exclusively for our seasonal menus. Guests can enjoy a full tasting menu, a five-course menu, or a bar menu. The bar also offers à la 

In [26]:
import gradio as gr

In [ ]:
# --- Define the Function for Gradio ---

# This function takes the user's input, runs the chain, and formats the output
# Ensure the `qa_chain` variable is accessible in this scope.
def ask_elevenmadison_assistant(user_query):
    """
    Processes the user query using the RAG chain and returns formatted results.
    """
    print(f"\nProcessing Gradio query: '{user_query}'")
    if not user_query or user_query.strip() == "":
        print("--> Empty query received.")
        return "Please enter a question.", ""  # Handle empty input gracefully

    try:
        # Run the query through our RAG chain
        result = qa_chain.invoke({"input": user_query})

        # Extract answer and sources
        answer = result.get("answer", "Sorry, I couldn't find an answer in the provided documents.")

        # ✅ Extract sources from context instead of result["sources"]
        context_docs = result.get("context", [])
        sources = list(set(
            doc.metadata.get("source", "Unknown Source")
            for doc in context_docs
            if doc.page_content.strip()  # skip empty docs
        ))
        sources_text = ", ".join(sources) if sources else "No specific sources identified."

        print(f"--> Answer generated: {answer[:100].strip()}...")
        print(f"--> Sources identified: {sources_text}")

        return answer.strip(), sources_text

    except Exception as e:
        error_message = f"An error occurred: {e}"
        print(f"--> Error during chain execution: {error_message}")
        return error_message, "Error occurred"


# --- Create the Gradio Interface using Blocks API ---
print("\nSetting up Gradio interface...")

with gr.Blocks(theme=gr.themes.Soft(), title="Eleven Madison Park Q&A Assistant") as demo:
    # Title and description for the app
    gr.Markdown(
        """
        # Eleven Madison Park - AI Q&A Assistant 💬
        Ask questions about the restaurant based on its website data.
        The AI provides answers and cites the source document.
        *(Examples: What are the menu prices? Who is the chef? Is it plant-based?)*
        """
    )

    # Input component for the user's question
    question_input = gr.Textbox(
        label = "Your Question:",
        placeholder = "e.g., What are the opening hours on Saturday?",
        lines = 2,  # Allow a bit more space for longer questions
    )

    # Row layout for the output components
    with gr.Row():
        # Output component for the generated answer (read-only)
        answer_output = gr.Textbox(label="Answer:", interactive=False, lines=6)  # User cannot edit this
        # Output component for the sources (read-only)
        sources_output = gr.Textbox(label="Sources:", interactive=False, lines=2)

    # Row for buttons
    with gr.Row():
        # Button to submit the question
        submit_button = gr.Button("Ask Question", variant="primary")
        # Clear button to reset inputs and outputs
        clear_button = gr.ClearButton(components=[question_input, answer_output, sources_output], value="Clear All")

    # Add some example questions for users to try
    gr.Examples(
        examples=[
            "What are the different menu options and prices?",
            "Who is the head chef?",
            "What is Magic Farms?",
            "Do I need a reservation for the bar?",
            "What is the dress code?",
            "Can I buy gift cards?"],
        inputs=question_input,  # Clicking example loads it into this input
        # We could potentially add outputs=[answer_output, sources_output] and cache examples
        # but that requires running the chain for each example beforehand.
        cache_examples=False,  # Don't pre-compute results for examples for simplicity
    )

    # --- Connect the Submit Button to the Function ---
    # When submit_button is clicked, call 'ask_emp_assistant'
    # Pass the value from 'question_input' as input
    # Put the returned values into 'answer_output' and 'sources_output' respectively
    submit_button.click(fn = ask_elevenmadison_assistant, inputs = question_input, outputs = [answer_output, sources_output])

print("Gradio interface defined.")

# --- Launch the Gradio App ---
print("\nLaunching Gradio app... (Stop the kernel or press Ctrl+C in terminal to quit)")
# demo.launch() # Launch locally in the notebook or browser
demo.launch()  


Setting up Gradio interface...
Gradio interface defined.

Launching Gradio app... (Stop the kernel or press Ctrl+C in terminal to quit)


C:\Users\princ\AppData\Local\Temp\ipykernel_37644\4106332948.py:44: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Eleven Madison Park Q&A Assistant") as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.



Processing Gradio query: 'Can I buy gift cards?'
--> Answer generated: System: Yes, you can purchase gift cards for Eleven Madison Park. Please note that all gift card s...
--> Sources identified: eleven_madison_park_data.txt
